# Classificação de Tickets de Incidentes com LLM: Livre, Zero-shot e Few-shot

**Estudo de Caso 2 — Minicurso "Inteligência Artificial aplicada à Resposta a Incidentes"**

## Objetivo

Este notebook compara **quatro formas de pedir a um LLM que classifique um incidente
de segurança**, usando dois arquivos CSV:

- `tickets_sinteticos.csv` — **24 tickets de incidentes** sintéticos, usados como base de avaliação.
- `tickets_fewshot_exemplos.csv` — **7 tickets sintéticos adicionais**, em 3 pares
  contrastantes + 1 exemplo híbrido, usados como exemplos rotulados nos prompts
  few-shot (nunca entram na avaliação).

Os quatro níveis, em ordem:

1. **Classificação livre (sem taxonomia)** — o modelo nomeia o incidente com suas
   próprias palavras, sem receber nenhuma lista de categorias. Mostra o problema que
   motiva os níveis seguintes: **dispersão semântica**.
2. **Zero-shot (com taxonomia)** — a tarefa e as 12 categorias possíveis, sem exemplos.
   É, estruturalmente, a abordagem que o **AutoClass** usa hoje: uma única chamada
   ao LLM, com contexto recuperado via RAG.
3. **Few-shot (2 exemplos, 1 par)** — usa um par de exemplos contrastantes para
   ensinar uma fronteira específica entre duas categorias que costumam ser confundidas.
4. **Few-shot (7 exemplos, 3 pares + 1 híbrido)** — cobre três fronteiras conceituais
   entre categorias adjacentes (CAT5/CAT3, CAT4/CAT6, CAT9/CAT12).

Medimos a **acurácia real** contra os rótulos dos CSVs — a mesma lógica de avaliação
usada em Severo et al. (2025) e Jesus Filho et al. (2025).

**Todos os tickets são fictícios** — nenhum dado real ou sensível é usado.

> **Pré-requisito:** coloque os dois arquivos CSV na mesma pasta deste notebook.

> **Sobre o AutoClass:** o código-fonte público
> ([github.com/gt-rnp-lfi/lfi-autoclass](https://github.com/gt-rnp-lfi/lfi-autoclass))
> mostra que a etapa de geração do AutoClass é, na arquitetura atual, uma **única
> chamada ao Gemini por classificação**, respondendo com base no contexto
> recuperado via RAG. A ferramenta está em atualização contínua da sua engenharia
> de prompts, e irá incorporar novas técnicas nas próximas versões.


## 1. Taxonomia de referência

As 12 categorias usadas neste notebook seguem a consolidação proposta por
Severo et al. (2025), com base nas diretrizes de tratamento de incidentes do
NIST SP 800-61 Rev. 3.


In [ ]:
TAXONOMIA = {
    "CAT1":  "Comprometimento de Conta — acesso não autorizado a contas de usuários ou administradores",
    "CAT2":  "Malware — infecção por código malicioso que compromete dispositivos ou dados",
    "CAT3":  "Negação de Serviço (DoS/DDoS) — indisponibilidade de sistemas ou redes",
    "CAT4":  "Exfiltração/Vazamento de Dados — acesso, cópia ou divulgação não autorizada de dados sensíveis",
    "CAT5":  "Exploração de Vulnerabilidade — uso de falhas conhecidas ou desconhecidas para comprometer ativos",
    "CAT6":  "Abuso Interno — ações intencionais ou negligentes de usuários internos",
    "CAT7":  "Engenharia Social — engano de pessoas para obter acesso ou informações",
    "CAT8":  "Incidente Físico ou de Infraestrutura — violação física que impacta ativos computacionais",
    "CAT9":  "Alteração Não Autorizada — modificação não autorizada em sistemas, dados ou configurações",
    "CAT10": "Uso Indevido de Recursos — uso não autorizado de sistemas para outros fins",
    "CAT11": "Problema de Fornecedor/Terceiro — incidente originado por falha de segurança de terceiros",
    "CAT12": "Tentativa de Intrusão — tentativas hostis de invasão ainda não confirmadas como bem-sucedidas",
}

for cod, desc in TAXONOMIA.items():
    print(f"{cod}: {desc}")


## 2. Carregando os tickets

`df` é o conjunto de **avaliação** (24 tickets — nenhum é excluído, já que os exemplos
few-shot vêm de um arquivo separado). `df_exemplos` traz os 7 tickets usados como
exemplos rotulados, organizados em pares:

| Par/exemplo | Tickets | Fronteira que ensina |
|---|---|---|
| CAT5 vs CAT3 | T025 / T026 | Vulnerabilidade (causa) vs. DDoS em andamento (efeito) |
| CAT4 vs CAT6 | T027 / T028 | Vazamento externo vs. abuso interno sem exportação |
| CAT9 vs CAT12 | T029 / T030 | Alteração consumada vs. tentativa de intrusão barrada |
| Híbrido CAT4 | T031 | Causa interna (erro de configuração) + efeito de exposição externa |


In [ ]:
import pandas as pd

df = pd.read_csv("tickets_sinteticos.csv")
df_exemplos = pd.read_csv("tickets_fewshot_exemplos.csv").set_index("id")

print(f"{len(df)} tickets de avaliação carregados.")
print(f"{len(df_exemplos)} tickets de exemplo carregados (few-shot).")


def _exemplo(tid):
    row = df_exemplos.loc[tid]
    return {"texto": row["texto"], "categoria": row["categoria"]}


PAR_CAT5_CAT3 = [_exemplo("T025"), _exemplo("T026")]
PAR_CAT4_CAT6 = [_exemplo("T027"), _exemplo("T028")]
PAR_CAT9_CAT12 = [_exemplo("T029"), _exemplo("T030")]
# T031: exemplo híbrido — causa interna (erro de configuração de um administrador)
# + efeito de exposição externa (achado por scanner de terceiros). Adicionado para
# testar se cobre o "meio do espectro" entre T027 (causa externa) e T028 (sem exposição).
EXEMPLO_HIBRIDO_CAT4 = [_exemplo("T031")]

EXEMPLOS_FEW_SHOT_2 = PAR_CAT5_CAT3
EXEMPLOS_FEW_SHOT_7 = PAR_CAT5_CAT3 + PAR_CAT4_CAT6 + PAR_CAT9_CAT12 + EXEMPLO_HIBRIDO_CAT4

print(f"\nFew-shot (2 exemplos): 1 par — CAT5/CAT3")
print(f"Few-shot (7 exemplos): 3 pares + 1 exemplo híbrido (T031, causa interna + exposição externa)")


## 3. Configuração da API (Gemini)

É necessário informar uma **chave de API do Google AI Studio**
(`GOOGLE_GENERATIVE_AI_API_KEY`, a mesma variável usada no AutoClass).

> **Sem chave configurada?** Use o notebook `02-classificacao_offline.ipynb`
> — plano B independente de API/internet, com os mesmos resultados reais.

> **Sobre o limite gratuito:** o plano gratuito do Gemini permite apenas
> **15 requisições por minuto** para o `gemini-3.1-flash-lite`. As células de
> classificação abaixo incluem uma pausa (~4,5s) entre chamadas e um retry
> automático em caso de erros transitórios (`429`, `503`, `500`, `502`, `504`).


In [ ]:
import os

# Opção 1: defina a variável de ambiente antes de abrir o Jupyter
#   export GOOGLE_GENERATIVE_AI_API_KEY="sua-chave-aqui"
#
# Opção 2: descomente a linha abaixo (apenas para teste local — nunca faça isso
# em notebooks versionados/compartilhados)
# os.environ["GOOGLE_GENERATIVE_AI_API_KEY"] = "sua-chave-aqui"

API_KEY = os.environ.get("GOOGLE_GENERATIVE_AI_API_KEY")
MODEL_NAME = os.environ.get("GOOGLE_GENERATIVE_MODEL", "gemini-3.1-flash-lite")

if API_KEY:
    # pip install google-genai
    from google import genai
    client = genai.Client(api_key=API_KEY)
    print(f"Cliente Gemini configurado. Modelo: {MODEL_NAME}")
else:
    client = None
    print("Nenhuma API key encontrada — use o notebook 02-classificacao_offline.ipynb para a demonstração.")


## 4. Funções auxiliares

Montagem de prompt (zero-shot e few-shot), chamada à API com retry/backoff, e
extração tolerante do código de categoria da resposta.


In [ ]:
import re
import time


def montar_prompt_zero_shot(ticket_texto: str, taxonomia: dict) -> str:
    categorias_str = "\n".join(f"- {cod}: {desc}" for cod, desc in taxonomia.items())
    return f"""Você é um analista de SOC especialista em categorização de incidentes de segurança.

Classifique o ticket abaixo em UMA das categorias a seguir, segundo o NIST SP 800-61 Rev. 3:

{categorias_str}

Ticket:
\"\"\"
{ticket_texto}
\"\"\"

Responda em formato JSON, apenas com as chaves "categoria" (o código, ex: CAT1) e "justificativa" (uma frase curta)."""


def montar_prompt_few_shot(ticket_texto: str, taxonomia: dict, exemplos: list) -> str:
    categorias_str = "\n".join(f"- {cod}: {desc}" for cod, desc in taxonomia.items())
    exemplos_str = "\n\n".join(
        f'Ticket: \"{ex["texto"][:300]}...\"\nResposta: {{"categoria": "{ex["categoria"]}"}}'
        for ex in exemplos
    )
    return f"""Você é um analista de SOC especialista em categorização de incidentes de segurança.

Classifique o ticket em UMA das categorias a seguir, segundo o NIST SP 800-61 Rev. 3:

{categorias_str}

Veja alguns exemplos já classificados:

{exemplos_str}

Agora classifique o novo ticket:
\"\"\"
{ticket_texto}
\"\"\"

Responda em formato JSON, apenas com as chaves "categoria" (o código, ex: CAT1) e "justificativa" (uma frase curta)."""


# Códigos de erro transitórios: vale tentar de novo (cota momentânea ou
# sobrecarga temporária do servidor do Google) — não são erros do nosso código.
CODIGOS_TRANSITORIOS = {
    "429": 20,  # RESOURCE_EXHAUSTED — cota por minuto excedida
    "503": 15,  # UNAVAILABLE — servidor sobrecarregado
    "500": 15,  # INTERNAL — erro interno transitório
    "502": 15,  # BAD_GATEWAY
    "504": 15,  # DEADLINE_EXCEEDED / gateway timeout
}


def classificar(prompt: str, max_tentativas: int = 5) -> str:
    if client is None:
        raise RuntimeError("Cliente Gemini não configurado — use o notebook 02-classificacao_offline.ipynb.")

    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            return resposta.text
        except Exception as e:
            erro_str = str(e)
            codigo_encontrado = next(
                (cod for cod in CODIGOS_TRANSITORIOS if cod in erro_str), None
            )
            if codigo_encontrado:
                base = CODIGOS_TRANSITORIOS[codigo_encontrado]
                espera = base * tentativa
                print(f"  [erro {codigo_encontrado} — aguardando {espera}s antes de tentar "
                      f"de novo ({tentativa}/{max_tentativas})]")
                time.sleep(espera)
            else:
                raise
    raise RuntimeError(
        f"Falhou após {max_tentativas} tentativas — pode ser cota esgotada ou "
        "instabilidade prolongada da API. Use o notebook 02-classificacao_offline.ipynb "
        "para continuar a demonstração."
    )


def extrair_categoria(resposta_texto: str):
    """Extrai o código CATx da resposta do modelo, tolerando variações de formatação."""
    match = re.search(r"CAT\d{1,2}", resposta_texto or "")
    return match.group(0) if match else None


SAMPLE_SIZE = 8  # quantos tickets processar em cada técnica, para comparação justa
# altere para len(df) para rodar os 24 tickets completos (mais lento, mas necessário
# para a análise de dispersão da Seção 5 e para a Seção 8.1)


## 5. Classificação livre (sem taxonomia) — por que definir uma taxonomia?

Antes de usar a taxonomia definida na Seção 1, vale perguntar: **ela é realmente
necessária?** E se não dermos categoria nenhuma ao modelo — só pedirmos para nomear o
incidente com as próprias palavras? Essa comparação recria, em miniatura, o desenho
experimental do Severo et al. (2025): categorização livre vs. taxonomia estruturada.

A expectativa da literatura é que a classificação livre produza **dispersão
semântica** — o mesmo tipo de incidente descrito com rótulos diferentes em tickets
diferentes, mesmo quando a categoria real (NIST) é idêntica. Vamos medir isso
diretamente: agrupar os tickets pela categoria NIST verdadeira (usada aqui só para
avaliação, nunca enviada ao modelo nesta seção) e contar quantos rótulos
**distintos** o modelo usou dentro de cada grupo.

Esse resultado motiva as seções seguintes: se a dispersão for alta, isso justifica
por que impor uma taxonomia fixa (Seção 6 em diante) vale a pena, mesmo que ela por
si só não garanta acurácia perfeita.

> Para essa análise fazer sentido, rode com `SAMPLE_SIZE = len(df)` (os 24 tickets
> completos) — com poucos tickets, dificilmente duas ocorrências da mesma categoria
> aparecem na amostra.


In [ ]:
def montar_prompt_livre(ticket_texto: str) -> str:
    return f"""Você é um analista de SOC. Leia o ticket abaixo e diga, com suas próprias palavras, que tipo de incidente de segurança é esse — não fornecemos nenhuma lista de categorias.

Ticket:
\"\"\"
{ticket_texto}
\"\"\"

Responda em formato JSON, apenas com as chaves "categoria" (um rótulo curto, 2 a 4 palavras, escolhido por você) e "justificativa" (uma frase curta)."""


def extrair_categoria_livre(resposta_texto: str) -> str:
    """Extrai o valor do campo 'categoria' de uma resposta livre (sem código CATx fixo)."""
    match = re.search(r'"categoria"\s*:\s*"([^"]+)"', resposta_texto or "")
    return match.group(1).strip() if match else (resposta_texto or "").strip()[:60]


In [ ]:
resultados_livre = []

for _, row in df.head(SAMPLE_SIZE).iterrows():
    prompt = montar_prompt_livre(row["texto"])
    saida = classificar(prompt)
    rotulo = extrair_categoria_livre(saida)
    resultados_livre.append({"id": row["id"], "esperado": row["categoria"], "rotulo_livre": rotulo})
    print(f'{row["id"]} (esperado={row["categoria"]}): "{rotulo}"')
    time.sleep(4.5)

df_livre = pd.DataFrame(resultados_livre)


In [ ]:
rotulos_unicos = df_livre["rotulo_livre"].nunique()
print(f"{rotulos_unicos} rótulos distintos usados para classificar {len(df_livre)} tickets, livremente.")
print(f"(A taxonomia estruturada usa apenas 12 categorias fixas.)\n")

print("--- Dispersão semântica: mesma categoria NIST verdadeira, rótulos livres diferentes ---")
for cat, grupo in df_livre.groupby("esperado"):
    if len(grupo) > 1:
        rotulos = grupo["rotulo_livre"].tolist()
        distintos = len(set(r.lower().strip() for r in rotulos))
        print(f"\n{cat} ({len(grupo)} tickets):")
        for tid, rot in zip(grupo["id"], rotulos):
            print(f'  {tid}: "{rot}"')
        print(f"  -> {distintos} rótulo(s) distinto(s) para {len(grupo)} ticket(s) da mesma categoria real")


### O que observar no resultado

- **Quantos rótulos distintos** apareceram no total, comparado às 12 categorias
  fixas da taxonomia? Quanto maior esse número, mais evidente a dispersão.
- **Dentro de cada grupo da mesma categoria NIST real**, os rótulos livres
  convergem (bom sinal — o modelo "sente" a mesma categoria mesmo sem taxonomia) ou
  divergem bastante (dispersão semântica, como descrito na literatura)?
- Essa dispersão é exatamente o problema prático que a taxonomia resolve: mesmo que
  o zero-shot com taxonomia (próxima seção) não acerte 100% dos tickets, pelo menos
  os rótulos serão **sempre os mesmos 12 códigos** — o que permite agregação,
  dashboards e métricas consistentes. A classificação livre não oferece essa
  garantia, independente da acurácia.


## 6. Classificação Zero-shot (com taxonomia)

É, estruturalmente, o que o AutoClass faz hoje: uma única chamada, sem exemplos.


In [ ]:
resultados_zero_shot = []

for _, row in df.head(SAMPLE_SIZE).iterrows():
    prompt = montar_prompt_zero_shot(row["texto"], TAXONOMIA)
    saida = classificar(prompt)
    predita = extrair_categoria(saida)
    correto = predita == row["categoria"]
    resultados_zero_shot.append(
        {"id": row["id"], "esperado": row["categoria"], "predito": predita, "correto": correto}
    )
    marcador = "✅" if correto else "❌"
    print(f"{row['id']}: esperado={row['categoria']} | predito={predita} {marcador}")
    time.sleep(4.5)

acertos = sum(r["correto"] for r in resultados_zero_shot)
print(f"\nAcurácia zero-shot: {acertos}/{len(resultados_zero_shot)} ({acertos/len(resultados_zero_shot):.1%})")


## 7. Classificação Few-shot (2 exemplos — 1 par)

Prompt com o par CAT5/CAT3 (T025 + T026). Esse par ensina uma distinção específica:
um relato pode mencionar "risco de amplificação de DDoS" e ainda assim ser, na
origem, uma vulnerabilidade de configuração — não um ataque em andamento.


In [ ]:
resultados_few_shot_2 = []

for _, row in df.head(SAMPLE_SIZE).iterrows():
    prompt = montar_prompt_few_shot(row["texto"], TAXONOMIA, EXEMPLOS_FEW_SHOT_2)
    saida = classificar(prompt)
    predita = extrair_categoria(saida)
    correto = predita == row["categoria"]
    resultados_few_shot_2.append(
        {"id": row["id"], "esperado": row["categoria"], "predito": predita, "correto": correto}
    )
    marcador = "✅" if correto else "❌"
    print(f"{row['id']}: esperado={row['categoria']} | predito={predita} {marcador}")
    time.sleep(4.5)

acertos = sum(r["correto"] for r in resultados_few_shot_2)
print(f"\nAcurácia few-shot (2 ex.): {acertos}/{len(resultados_few_shot_2)} ({acertos/len(resultados_few_shot_2):.1%})")


## 7.1 Classificação Few-shot (7 exemplos — 3 pares + 1 híbrido)

Prompt com os 3 pares mais um exemplo híbrido (T031): causa interna (erro de
configuração de um administrador) que resulta em exposição externa (achado por um
scanner de terceiros) — um caso que mistura os dois extremos do par CAT4/CAT6
(agente externo vs. agente interno).


In [ ]:
resultados_few_shot_7 = []

for _, row in df.head(SAMPLE_SIZE).iterrows():
    prompt = montar_prompt_few_shot(row["texto"], TAXONOMIA, EXEMPLOS_FEW_SHOT_7)
    saida = classificar(prompt)
    predita = extrair_categoria(saida)
    correto = predita == row["categoria"]
    resultados_few_shot_7.append(
        {"id": row["id"], "esperado": row["categoria"], "predito": predita, "correto": correto}
    )
    marcador = "✅" if correto else "❌"
    print(f"{row['id']}: esperado={row['categoria']} | predito={predita} {marcador}")
    time.sleep(4.5)

acertos = sum(r["correto"] for r in resultados_few_shot_7)
print(f"\nAcurácia few-shot (7 ex.): {acertos}/{len(resultados_few_shot_7)} ({acertos/len(resultados_few_shot_7):.1%})")


## 8. Comparando as três estratégias lado a lado

In [ ]:
comparacao = pd.DataFrame(resultados_zero_shot)[["id", "esperado", "predito"]].rename(
    columns={"predito": "zero_shot"}
)
comparacao["few_shot_2"] = pd.DataFrame(resultados_few_shot_2)["predito"].values
comparacao["few_shot_7"] = pd.DataFrame(resultados_few_shot_7)["predito"].values

for col in ["zero_shot", "few_shot_2", "few_shot_7"]:
    comparacao[f"{col}_ok"] = comparacao[col] == comparacao["esperado"]

comparacao


## 8.1 Salvando os resultados

Rode esta célula depois de executar as Seções 5, 6, 7 e 7.1 com `SAMPLE_SIZE = len(df)`
(os 24 tickets completos). Isso salva um `resultados_completos.csv` com a avaliação
inteira, mesmo que você tenha processado só uma amostra menor por questão de tempo.


In [ ]:
comparacao.to_csv("resultados_completos.csv", index=False)
print(f"Salvo em resultados_completos.csv ({len(comparacao)} tickets).")

for col in ["zero_shot", "few_shot_2", "few_shot_7"]:
    acc = comparacao[f"{col}_ok"].mean()
    print(f"Acurácia {col}: {acc:.1%}")

divergentes = comparacao[
    comparacao[["zero_shot", "few_shot_2", "few_shot_7"]].nunique(axis=1) > 1
]
print(f"\n{len(divergentes)} ticket(s) com respostas divergentes entre as estratégias:")
divergentes


## 8.2 Matriz de confusão

Com 24 tickets distribuídos por até 12 categorias, a maioria das células desta
matriz terá 0, 1 ou 2 ocorrências — não é uma amostra estatisticamente robusta
para revelar padrões sistemáticos de confusão. Mesmo assim, ela é útil para
visualizar concretamente os erros observados nas seções anteriores (ex.: CAT5
confundido com CAT12 ou CAT3, CAT4 confundido com CAT6).


In [ ]:
import matplotlib.pyplot as plt

def plotar_matriz_confusao(df_comp: pd.DataFrame, coluna_predita: str, titulo: str):
    categorias = sorted(
        set(df_comp["esperado"]) | set(df_comp[coluna_predita].dropna()),
        key=lambda c: int(c.replace("CAT", "")),
    )
    matriz = pd.crosstab(df_comp["esperado"], df_comp[coluna_predita]).reindex(
        index=categorias, columns=categorias, fill_value=0
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(matriz.values, cmap="Reds")

    ax.set_xticks(range(len(categorias)))
    ax.set_yticks(range(len(categorias)))
    ax.set_xticklabels(categorias, rotation=45, ha="right")
    ax.set_yticklabels(categorias)
    ax.set_xlabel("Categoria predita")
    ax.set_ylabel("Categoria esperada")
    ax.set_title(titulo, fontsize=12, pad=12)

    for i in range(len(categorias)):
        for j in range(len(categorias)):
            valor = matriz.values[i, j]
            if valor > 0:
                cor_texto = "white" if valor > matriz.values.max() / 2 else "black"
                ax.text(j, i, str(valor), ha="center", va="center", color=cor_texto, fontsize=9)

    plt.tight_layout()
    plt.show()


plotar_matriz_confusao(comparacao, "zero_shot", "Zero-shot — matriz de confusão")
plotar_matriz_confusao(comparacao, "few_shot_7", "Few-shot (7 exemplos) — matriz de confusão")


## 8.3 Gerando o arquivo para a Prática 3

O resultado final desta prática — a categoria atribuída a cada ticket pela
melhor estratégia avaliada (few-shot com 7 exemplos) — é salvo em
`saida/incidentes_classificados.csv`. Esse arquivo alimenta a Prática 3
(geração de playbooks).


In [ ]:
import os

os.makedirs("saida", exist_ok=True)

saida_pratica3 = df.merge(
    comparacao[["id", "few_shot_7"]].rename(columns={"few_shot_7": "categoria_atribuida"}),
    on="id",
)[["id", "id_incidente", "texto", "categoria_atribuida"]]

saida_pratica3.to_csv("saida/incidentes_classificados.csv", index=False)
print(f"Salvo em saida/incidentes_classificados.csv ({len(saida_pratica3)} tickets).")
saida_pratica3.head()


## 9. Sem acesso à API?

Use o notebook **`02-classificacao_offline.ipynb`** — uma versão totalmente
independente (sem CSV externo, sem chave de API, sem internet), com os mesmos
resultados reais deste material já pré-computados e visualizados (gráfico de
acurácia, tabelas coloridas por ticket). Ele já vem com os outputs prontos, então
funciona mesmo sem executar nenhuma célula.


## 10. Discussão

### Por que o zero-shot deste notebook representa o AutoClass

Na arquitetura atual, o AutoClass faz uma única chamada ao LLM por classificação,
com contexto recuperado via RAG. É exatamente o que o nível de zero-shot deste
notebook reproduz — por isso ele serve como referência de comparação para os
demais níveis. A ferramenta está em atualização contínua e irá incorporar novas
técnicas de prompt ao longo do tempo.

### Sobre os exemplos few-shot

Os exemplos usados aqui vêm de um CSV separado (`tickets_fewshot_exemplos.csv`), com
7 tickets: 3 pares contrastantes — cada par ensina uma fronteira conceitual entre
categorias historicamente confundidas (CAT5/CAT3, CAT4/CAT6, CAT9/CAT12) — mais 1
exemplo híbrido (T031). Nenhum deles replica a situação específica de um ticket do
conjunto de avaliação, o que evita que o modelo simplesmente "decore a resposta" em
vez de generalizar.

### O que os resultados mostram — uma progressão de 87,5% para 95,8%

- **O par CAT5/CAT3 corrige dois tickets de uma vez.** Com apenas 2 exemplos, a
  acurácia sobe de 87,5% para 91,7%. Os dois tickets corrigidos envolvem o mesmo
  tipo de confusão: um relato menciona risco de amplificação de DDoS, mas a causa
  raiz é uma vulnerabilidade de configuração, não um ataque em andamento.
- **Nem todo par cobre todos os casos.** O par CAT4/CAT6 (vazamento vs. abuso
  interno) não corrige, sozinho, um ticket que combina os dois: causa **interna**
  (erro de um funcionário) com efeito de exposição **externa** (achado por um
  scanner). Um exemplo híbrido, desenhado para essa combinação intermediária,
  resolve o caso — levando a acurácia a 95,8%.
- **Um ticket resiste a todas as três estratégias.** O relato correspondente é vago
  ("atividade incomum no servidor", sem detalhes técnicos) — evidência de que o
  problema ali é a qualidade do relato de origem, não a estratégia de prompt.
  Nenhuma engenharia de prompt recupera informação que nunca esteve no texto
  original. Essa observação conecta com a discussão sobre qualidade do alerta de
  entrada (Alahmadi et al.).

### O que isso ensina sobre few-shot

Cada exemplo adicionado corrigiu exatamente o tipo de erro para o qual foi
desenhado, sem introduzir erros novos em outros tickets. Exemplos bem curados
atacam ambiguidades específicas e conhecidas — mas não substituem a necessidade de
um relato de qualidade mínima na entrada.

### Taxonomia resolve consistência, few-shot resolve acurácia

A classificação livre (Seção 5) mostra por que uma taxonomia estruturada importa:
sem ela, o mesmo tipo de incidente é descrito com rótulos diferentes a cada vez,
mesmo quando a categoria real é idêntica — o que inviabiliza métricas e dashboards
consistentes.

Juntando os quatro níveis, a tese central deste Estudo de Caso fica em três camadas:

1. **Classificação livre** — sem consistência de rótulos entre tickets
   semanticamente equivalentes.
2. **Taxonomia fixa (zero-shot, 87,5%)** — resolve a consistência, mas ainda erra
   alguns casos.
3. **Taxonomia + few-shot bem curado (95,8%)** — mantém a consistência **e** reduz o
   erro residual, atacando ambiguidades específicas.

É essa progressão — de "sem estrutura" para "estrutura simples" para "estrutura
refinada" — que explica por que o AutoClass usa uma taxonomia fixa, e não
categorização livre.

### Referências

- Categorias de incidentes: taxonomia de 12 categorias adaptada do NIST SP 800-61 Rev. 3 por Severo et al. (2025).
- Estratégias de prompting (zero-shot/few-shot): Brown et al. (2020), *Language Models are Few-Shot Learners*.
- Ganhos de acurácia em tickets reais de segurança: Jesus Filho et al. (2025); Severo et al. (2025).
- Panorama de técnicas de engenharia de prompts: Schulhoff et al. (2024), *The Prompt Report*.
- Arquitetura RAG do AutoClass: gt-rnp-lfi/lfi-autoclass (GitHub).

### Próximos passos

Este notebook cobre o Nível 1 (classificação via API) do Estudo de Caso 2. Os
próximos níveis aprofundam:

- **Nível 2** — comparação entre classificação via API (Gemini) e modelos locais
  (ex: LLaMA, Mistral), reusando os mesmos CSVs como base de comparação.
- **Nível 3** — demonstração da ferramenta AutoClass (LFI) em produção, mostrando o
  pipeline completo de upload, categorização e geração de playbooks.
